# 02 - Pipeline Spark et indicateurs Gold

Ce notebook documente le job `spark/gold_job.py` : choix de jointure, window functions, TF-IDF, score d'impact et resultats dans Elasticsearch (`arxiv-papers-enriched`).

Prerequis : DAGs `20_silver`, `30_index_arxiv_papers` et `31_gold_spark` termines.

In [ ]:
%pip install -q pandas pyarrow boto3 matplotlib requests "elasticsearch==8.13.2" scipy
import io, json, re, sys, collections
import boto3, pandas as pd, matplotlib.pyplot as plt, requests
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
sys.path.insert(0, "../src")          # modules de normalisation du projet (purs, sans dependance Docker)

# Acces MinIO depuis votre machine (la stack Docker doit tourner)
s3 = boto3.client("s3", endpoint_url="http://localhost:9000", aws_access_key_id="minioadmin",
                  aws_secret_access_key="minioadmin123", region_name="us-east-1")

def list_keys(bucket, prefix=""):
    out, token = [], None
    while True:
        kw = dict(Bucket=bucket, Prefix=prefix, **({"ContinuationToken": token} if token else {}))
        r = s3.list_objects_v2(**kw); out += [o["Key"] for o in r.get("Contents", [])]
        if not r.get("IsTruncated"): return out
        token = r["NextContinuationToken"]

def read_jsonl(bucket, keys):
    rows = []
    for k in keys:
        body = s3.get_object(Bucket=bucket, Key=k)["Body"].read().decode("utf-8")
        rows += [json.loads(l) for l in body.splitlines() if l]
    return pd.DataFrame(rows)

def latest_partition(bucket):
    keys = sorted(k for k in list_keys(bucket) if k.endswith(".jsonl") and k.startswith("dt="))
    return [k for k in keys if k.startswith(keys[-1].split("/")[0] + "/")]

def read_silver(bucket):
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=bucket, Key="data/part-0000.parquet")["Body"].read()))

def read_report(bucket, key="_reports/latest.json"):
    return json.loads(s3.get_object(Bucket=bucket, Key=key)["Body"].read())

from elasticsearch import Elasticsearch, helpers
es = Elasticsearch("http://localhost:9200")
gold = pd.DataFrame(d["_source"] for d in helpers.scan(es, index="arxiv-papers-enriched", query={"query": {"match_all": {}}}, size=2000))
for c in ("submitted_at", "first_hn_at"): gold[c] = pd.to_datetime(gold[c], utc=True, errors="coerce")
rep = read_report("arxiv-clean", "_reports/gold_run.json")
print(f"{len(gold):,} documents dans arxiv-papers-enriched | ecriture : {rep.get('es_write_method')}")

## 1. Jointure de trois sources

`arxiv` est la **table de reference** (une ligne par article) ; citations et Hacker News sont rattachees par **jointure gauche** sur `arxiv_id` : un article sans citation ni mention HN reste dans le resultat. Hacker News est d'abord **agrege par article** (nombre de posts, score total/max, commentaires, premiere apparition) pour qu'une jointure ne multiplie jamais les lignes.

In [ ]:
cov = rep["coverage"]
display(pd.DataFrame([{"mesure": k, "articles": v, "% du total": round(100 * v / cov["arxiv_papers"], 2)} for k, v in cov.items()]))

## 2. Strategie de jointure : sort-merge vs broadcast (`.explain()`)

In [ ]:
js = rep["join_strategies"]
display(pd.DataFrame([{"strategie": k, "duree_s": v["seconds"], "lignes": v["rows"]} for k, v in js.items()]))
for k, v in js.items():
    print(f"\n===== plan physique : {k} =====")
    print("\n".join(l for l in v["plan"].splitlines() if any(t in l for t in ("Join", "Exchange", "Broadcast", "Scan"))))

**Lecture** : en *sort-merge*, Spark redistribue (`Exchange`) et trie les deux cotes de chaque jointure. En *broadcast*, les petites tables (citations amincies, agregat HN) sont envoyees a tous les executeurs et la grande table n'est pas redistribuee : plus rapide tant que la petite table tient en memoire. Le job retient le broadcast ; sur le corpus complet (2,7 M d'articles), les citations ne tiendraient plus en broadcast et il faudrait repasser en sort-merge.

## 3. Score d'impact

`impact_score = 100 x ( 0,50 x C + 0,30 x H + 0,20 x R )`

| Composante | Formule | Justification |
|---|---|---|
| **C** citations | `log1p(citations) / log1p(p99)` | signal scientifique valide par les pairs ; poids le plus fort |
| **H** Hacker News | `log1p(score + commentaires) / log1p(p99)` | attention de la communaute technique, signal precoce |
| **R** fraicheur | `0,5 ^ (age / 3 ans)` | un article recent n'a pas eu le temps d'etre cite ; evite que les anciens dominent |

Echelle logarithmique (distributions en loi de puissance) et plafonnement au 99e percentile (un article aberrant n'ecrase pas l'echelle). Limite assumee : un article sans donnee OpenAlex a `C = 0` (drapeau `has_citation_data`).

In [ ]:
print("Seuils utilises :", rep["thresholds"], "| poids :", rep["weights"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
gold["impact_score"].plot.hist(bins=50, ax=ax[0], title="Distribution du score d'impact")
gold[["citation_component", "hn_component", "recency_component"]].mean().plot.bar(ax=ax[1], title="Valeur moyenne de chaque composante")
plt.tight_layout(); plt.show()
cols = ["arxiv_id", "title", "primary_category", "year", "cited_by_count", "hn_engagement", "impact_score"]
display(gold.sort_values("impact_score", ascending=False)[cols].head(15))

## 4. Window functions : croissance et velocite par categorie

In [ ]:
g = gold.drop_duplicates(["primary_category", "year"])[["primary_category", "year", "category_year_count", "category_growth_yoy"]]
top = gold["primary_category"].value_counts().head(6).index
pivot = g[g.primary_category.isin(top)].pivot(index="year", columns="primary_category", values="category_year_count")
pivot.plot(marker="o", figsize=(10, 4.5), title="Articles par categorie et par annee (echantillon)"); plt.show()
display(g[g.primary_category.isin(top)].sort_values(["primary_category", "year"]).tail(12))
print("`category_growth_yoy` = (n(annee) - n(annee-1)) / n(annee-1), calcule avec lag() sur une fenetre par categorie.")
print("`velocity_pct_in_category` = percent_rank() de la velocite de citation (citations des 2 dernieres annees / age plafonne a 2) dans la categorie.")

## 5. TF-IDF : termes distinctifs

In [ ]:
terms = gold.explode("top_terms").dropna(subset=["top_terms"])
for cat in top[:4]:
    print(f"{cat:8s}", terms[terms.primary_category == cat]["top_terms"].value_counts().head(12).index.tolist())
display(gold[["title", "top_terms"]].sample(5, random_state=3))

## 6. Cas d'usage : ce que seule la combinaison des trois sources revele

In [ ]:
cols = ["arxiv_id", "title", "primary_category", "year", "cited_by_count", "hn_engagement", "impact_score"]
star = gold[gold.academic_star_no_hn]; viral = gold[gold.viral_low_citation]
print(f"Tres cites (top 10 %) mais jamais discutes sur HN : {len(star):,} articles"); display(star.sort_values("cited_by_count", ascending=False)[cols].head(8))
print(f"Viraux sur HN (top 25 %) mais peu cites (<= mediane) : {len(viral):,} articles"); display(viral.sort_values("hn_engagement", ascending=False)[cols].head(8))
both = gold[gold.has_hn & gold.has_citation_data]
if len(both) > 5:
    from scipy.stats import spearmanr
    rho, p = spearmanr(both["cited_by_count"], both["hn_engagement"])
    print(f"Correlation de Spearman citations x engagement HN (n={len(both)}) : rho = {rho:.2f} (p = {p:.3g})")
    both.plot.scatter(x="cited_by_count", y="hn_engagement", loglog=True, figsize=(6, 4.5), title="Citations vs engagement HN"); plt.show()